# Source-Free Adaptation for IoT Intrusion Detection — Feasibility (Phase 0)
**Question:** when an IDS trained on one network is deployed on another, and only the *trained model* (no source data) plus *unlabeled* target traffic is available, do existing source-free adaptation (SFDA) methods recover the lost performance — or do they fail in IDS-specific ways?

**Data:** NetFlow-v2 datasets (Sarhan et al., University of Queensland), which share the same 43 NetFlow features by design: NF-ToN-IoT-v2 (≈64% attacks) and NF-UNSW-NB15-v2 (≈4% attacks). IPs, ports and DNS query IDs are dropped (identifiers).

**Methods compared on the target:** Source-only · BN-Adapt (target batch-norm statistics) · TENT (entropy minimisation on BN parameters, Wang et al., ICLR 2021) · SHOT (information maximisation + self-supervised pseudo-labels, Liang et al., ICML 2020) · Oracle (supervised on labelled target — upper bound).

**Target attack prior** is varied (natural, 5%, 50%) because real deployments are benign-dominated and SFDA losses make prior assumptions.

**Decision rule (printed at the end):** GO only if (1) the cross-network gap is large (Oracle − Source-only ≥ 0.15 macro-F1) **and** (2) existing SFDA fails measurably (recovers < 50% of the gap, or falls below Source-only, or misses unseen attack types).

Results go to Drive (`sfda_ids_phase0`) and are **resumable** (`Run all` again after a disconnect).

In [ ]:
# ===== 1. Setup =====
import os, glob, json, time, math, random, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score

try:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT = '/content/drive/MyDrive'
except Exception:
    ROOT = '.'
OUT = f'{ROOT}/sfda_ids_phase0'; os.makedirs(OUT, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| results ->', OUT)

CFG = dict(
    datasets={'ton': 'dhoogla/nftoniotv2', 'unsw': 'dhoogla/nfunswnb15v2'},
    known_rows={'ton': 16_940_496, 'unsw': 2_390_275},
    n_per_dataset=300_000, test_frac=0.3, seeds=[0, 1, 2],
    hidden=[128, 64], src_epochs=5, bs=512, lr=1e-3,
    priors=['natural', 0.05, 0.5], n_target_train=100_000, n_target_test=40_000,
    tent_epochs=1, tent_lr=1e-3, shot_epochs=3, shot_lr=1e-4, shot_beta=0.3,
    gap_min=0.15, recovery_min=0.5, harm_margin=0.05, unseen_recall_min=0.5)
json.dump(CFG, open(f'{OUT}/config.json', 'w'), indent=2)
DROP = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'DNS_QUERY_ID', 'Dataset']

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ===== 2. Data utilities =====
def find_files(root):
    fs = sorted(glob.glob(os.path.join(root, '**', '*.parquet'), recursive=True))
    return fs if fs else sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True))

def load_nf(key, slug, n, seed=0):
    cache = f'{OUT}/data_{key}_{n}.parquet'
    if os.path.exists(cache):
        print(key, 'loaded from cache'); return pd.read_parquet(cache)
    import kagglehub
    root = kagglehub.dataset_download(slug); files = find_files(root)
    assert files, f'no parquet/csv files found under {root}'
    print(key, 'files:', [os.path.basename(f) for f in files])
    rs = np.random.RandomState(seed); parts = []
    if files[0].endswith('.parquet'):
        import pyarrow.parquet as pq
        pfs = [pq.ParquetFile(f) for f in files]; total = sum(p.metadata.num_rows for p in pfs)
        frac = min(1.0, 1.3 * n / total)
        for p in pfs:
            for b in p.iter_batches(batch_size=1_000_000):
                parts.append(b.to_pandas().sample(frac=frac, random_state=rs))
    else:
        frac = min(1.0, 1.3 * n / CFG['known_rows'][key])
        for f in files:
            for d in pd.read_csv(f, chunksize=1_000_000, low_memory=False):
                parts.append(d.sample(frac=frac, random_state=rs))
    df = pd.concat(parts, ignore_index=True)
    if len(df) > n: df = df.sample(n=n, random_state=seed).reset_index(drop=True)
    try: df.to_parquet(cache)
    except Exception as e: print('cache not written:', e)
    print(key, 'sampled', len(df), 'rows'); return df

def prepare(df):
    cols = {c.lower(): c for c in df.columns}
    lab, att = cols.get('label'), cols.get('attack')
    assert lab is not None, f'no Label column in {list(df.columns)}'
    s = df[lab].astype(str).str.strip().str.lower()
    y = (~s.isin(['0', '0.0', 'benign', 'false', 'normal'])).astype(np.int64).values
    a = df[att].astype(str).str.strip().str.lower().values if att else np.where(y == 1, 'attack', 'benign')
    X = df.drop(columns=[c for c in [lab, att] + DROP if c is not None and c in df.columns])
    X = X.apply(lambda c: pd.to_numeric(c.astype(str) if str(c.dtype) in ('category', 'object', 'bool') else c, errors='coerce'))
    X = X.loc[:, X.notna().any()].fillna(0.0)
    return X, y, a

def log_t(X): return (np.sign(X) * np.log1p(np.abs(X))).astype(np.float32)

def build_data(raw):
    prep = {k: prepare(v) for k, v in raw.items()}
    common = sorted(set.intersection(*[set(p[0].columns) for p in prep.values()]))
    print(f'{len(common)} common numeric features')
    data = {}
    for k, (X, y, a) in prep.items():
        Xl = log_t(X[common].values)
        idx = np.random.RandomState(0).permutation(len(y)); ntest = int(CFG['test_frac'] * len(y))
        te, tr = idx[:ntest], idx[ntest:]
        data[k] = dict(Xtr=Xl[tr], ytr=y[tr], atr=a[tr], Xte=Xl[te], yte=y[te], ate=a[te])
        print(f'{k}: train {len(tr)} test {len(te)} attack rate {y.mean():.3f} | types {sorted(set(a[y == 1]))}')
    return data, common

In [ ]:
# ===== 3. Model, training, SFDA methods, metrics =====
class Net(nn.Module):
    def __init__(s, d, h, k=2):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(d, h[0]), nn.BatchNorm1d(h[0]), nn.ReLU(),
                            nn.Linear(h[0], h[1]), nn.BatchNorm1d(h[1]), nn.ReLU())
        s.h = nn.Linear(h[1], k)
    def forward(s, x): return s.h(s.f(x))

def T(x): return torch.as_tensor(x, device=device)

def train_sup(X, y, seed, epochs):
    set_seed(seed); m = Net(X.shape[1], CFG['hidden']).to(device); opt = torch.optim.Adam(m.parameters(), CFG['lr'])
    Xt, yt = T(X), T(y); g = torch.Generator(device='cpu').manual_seed(seed)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(Xt), generator=g).to(device)
        for i in range(0, len(perm), CFG['bs']):
            b = perm[i:i + CFG['bs']]
            if len(b) < 2: continue
            opt.zero_grad(); F.cross_entropy(m(Xt[b]), yt[b]).backward(); opt.step()
    return m.eval()

@torch.no_grad()
def predict(m, X):
    m.eval(); Xt = T(X)
    return torch.cat([m(Xt[i:i + 8192]).argmax(1) for i in range(0, len(Xt), 8192)]).cpu().numpy()

def bn_adapt(m, X):
    m = copy.deepcopy(m)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d): mod.reset_running_stats(); mod.momentum = None
    m.train(); Xt = T(X)
    with torch.no_grad():
        for i in range(0, len(Xt), 1024):
            if len(Xt[i:i + 1024]) > 1: m(Xt[i:i + 1024])
    return m.eval()

def tent(m, X, seed):
    set_seed(seed); m = copy.deepcopy(m); m.requires_grad_(False); params = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm1d):
            mod.requires_grad_(True); params += [mod.weight, mod.bias]
    opt = torch.optim.Adam(params, CFG['tent_lr']); Xt = T(X)
    for _ in range(CFG['tent_epochs']):
        m.train(); perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            if len(b) < 2: continue
            lp = F.log_softmax(m(Xt[b]), 1); loss = -(lp.exp() * lp).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def shot(m, X, seed):
    set_seed(seed); m = copy.deepcopy(m); m.h.requires_grad_(False)
    opt = torch.optim.Adam(m.f.parameters(), CFG['shot_lr']); Xt = T(X)
    for _ in range(CFG['shot_epochs']):
        m.eval()
        with torch.no_grad():
            feats = torch.cat([m.f(Xt[i:i + 8192]) for i in range(0, len(Xt), 8192)])
            probs = F.softmax(m.h(feats), 1); fn = F.normalize(feats, dim=1)
            c = F.normalize((probs.T @ fn) / (probs.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
            oh = F.one_hot(pl, 2).float()
            c = F.normalize((oh.T @ fn) / (oh.sum(0)[:, None] + 1e-8), dim=1); pl = (fn @ c.T).argmax(1)
        m.train(); perm = torch.randperm(len(Xt), device=device)
        for i in range(0, len(perm), 256):
            b = perm[i:i + 256]
            if len(b) < 2: continue
            out = m(Xt[b]); p = F.softmax(out, 1)
            ent = -(p * torch.log(p + 1e-5)).sum(1).mean(); pbar = p.mean(0); div = (pbar * torch.log(pbar + 1e-5)).sum()
            loss = ent + div + CFG['shot_beta'] * F.cross_entropy(out, pl[b])
            opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()

def metrics(y, pred):
    pos, neg = y == 1, y == 0
    tpr = ((pred == 1) & pos).sum() / max(pos.sum(), 1); fpr = ((pred == 1) & neg).sum() / max(neg.sum(), 1)
    return dict(macro_f1=f1_score(y, pred, average='macro', zero_division=0), attack_recall=float(tpr), fpr=float(fpr),
                bal_acc=float(0.5 * (tpr + 1 - fpr)), true_attack_rate=float(y.mean()), pred_attack_rate=float(pred.mean()))

def resample(X, y, a, prior, n, seed):
    rs = np.random.RandomState(seed)
    if prior == 'natural':
        idx = rs.permutation(len(y))[:n]; return X[idx], y[idx], a[idx]
    att, ben = np.where(y == 1)[0], np.where(y == 0)[0]
    n = int(min(n, len(att) / prior, len(ben) / (1 - prior)))
    na = int(round(prior * n)); idx = np.concatenate([rs.choice(att, na, replace=False), rs.choice(ben, n - na, replace=False)])
    idx = rs.permutation(idx); return X[idx], y[idx], a[idx]

In [ ]:
# ===== 4. Experiment runner (resumable) =====
def run_all(data, pairs, out_prefix, seeds, src_epochs):
    csv_m, csv_t = f'{out_prefix}_metrics.csv', f'{out_prefix}_per_type.csv'
    done = set()
    if os.path.exists(csv_m):
        d = pd.read_csv(csv_m); done = set(zip(d.seed, d.src, d.tgt, d.prior.astype(str)))
    for seed in seeds:
        for s, t in pairs:
            todo = [p for p in CFG['priors'] if (seed, s, t, str(p)) not in done]
            if not todo: continue
            S, Tg = data[s], data[t]
            mu, sd = S['Xtr'].mean(0), S['Xtr'].std(0) + 1e-6
            norm = lambda X: np.clip((X - mu) / sd, -10, 10).astype(np.float32)
            t0 = time.time(); src = train_sup(norm(S['Xtr']), S['ytr'], seed, src_epochs)
            in_dom = metrics(S['yte'], predict(src, norm(S['Xte'])))['macro_f1']
            seen = set(S['atr'][S['ytr'] == 1])
            for prior in todo:
                Xa, ya_, aa = resample(Tg['Xtr'], Tg['ytr'], Tg['atr'], prior, CFG['n_target_train'], seed)
                Xb, yb_, ab = resample(Tg['Xte'], Tg['yte'], Tg['ate'], prior, CFG['n_target_test'], seed + 100)
                Xa, Xb = norm(Xa), norm(Xb)
                models = {'source_only': src, 'bn_adapt': bn_adapt(src, Xa), 'tent': tent(src, Xa, seed), 'shot': shot(src, Xa, seed),
                          'oracle': train_sup(Xa, ya_, seed, src_epochs)}
                rows, trows = [], []
                for name, m in models.items():
                    pred = predict(m, Xb)
                    rows.append(dict(seed=seed, src=s, tgt=t, prior=str(prior), method=name, src_in_domain_f1=in_dom, **metrics(yb_, pred)))
                    for typ in sorted(set(ab[yb_ == 1])):
                        msk = ab == typ
                        trows.append(dict(seed=seed, src=s, tgt=t, prior=str(prior), method=name, attack_type=typ,
                                          seen_in_source=typ in seen, n=int(msk.sum()), recall=float((pred[msk] == 1).mean())))
                pd.DataFrame(rows).to_csv(csv_m, mode='a', header=not os.path.exists(csv_m), index=False)
                pd.DataFrame(trows).to_csv(csv_t, mode='a', header=not os.path.exists(csv_t), index=False)
                r = {x['method']: round(x['macro_f1'], 3) for x in rows}
                print(f'seed {seed} {s}->{t} prior {prior}: {r}  ({(time.time()-t0)/60:.1f} min)', flush=True)
    return pd.read_csv(csv_m), pd.read_csv(csv_t)

## Self-test on synthetic data (≈ 30 s) — must finish without errors before the real run

In [ ]:
# ===== 5. Self-test =====
def synth(n, shift, prior, seed):
    rs = np.random.RandomState(seed); y = (rs.rand(n) < prior).astype(np.int64)
    X = rs.randn(n, 20).astype(np.float32) + y[:, None] * 1.5 + shift
    a = np.where(y == 1, np.where(rs.rand(n) < 0.5, 'dos', 'scan'), 'benign')
    return X, y, a
fake = {}
for k, (sh, pr) in {'A': (0.0, 0.6), 'B': (0.8, 0.1)}.items():
    X, y, a = synth(6000, sh, pr, len(k)); te = slice(0, 2000); tr = slice(2000, None)
    fake[k] = dict(Xtr=X[tr], ytr=y[tr], atr=a[tr], Xte=X[te], yte=y[te], ate=a[te])
_save = dict(CFG); CFG.update(n_target_train=2000, n_target_test=1000)
for f in glob.glob(f'{OUT}/selftest_*'): os.remove(f)
mt, tt = run_all(fake, [('A', 'B')], f'{OUT}/selftest', seeds=[0], src_epochs=2)
CFG.clear(); CFG.update(_save)
assert len(mt) == 3 * 5 and len(tt) > 0, 'self-test produced incomplete results'
print('SELF-TEST PASSED'); print(mt[['prior', 'method', 'macro_f1', 'pred_attack_rate', 'true_attack_rate']].round(3).to_string(index=False))

## Real data

In [ ]:
# ===== 6. Load NetFlow-v2 datasets (cached on Drive after the first run) =====
raw = {k: load_nf(k, slug, CFG['n_per_dataset']) for k, slug in CFG['datasets'].items()}
DATA, FEATS = build_data(raw); del raw
json.dump(FEATS, open(f'{OUT}/features.json', 'w'))

In [ ]:
# ===== 7. Run (3 seeds x 2 directions x 3 priors x 5 methods) =====
keys = list(CFG['datasets'])
PAIRS = [(s, t) for s in keys for t in keys if s != t]
MT, TT = run_all(DATA, PAIRS, f'{OUT}/real', CFG['seeds'], CFG['src_epochs'])
print('done:', len(MT), 'metric rows')

In [ ]:
# ===== 8. Analysis and GO / NO-GO =====
MT = pd.read_csv(f'{OUT}/real_metrics.csv'); TT = pd.read_csv(f'{OUT}/real_per_type.csv')
cols = ['macro_f1', 'attack_recall', 'fpr', 'pred_attack_rate', 'true_attack_rate']
summ = MT.groupby(['src', 'tgt', 'prior', 'method'])[cols].agg(['mean', 'std']).round(3)
summ.to_csv(f'{OUT}/summary.csv'); print(summ.to_string())
print('\nSource in-domain macro-F1:', MT.groupby('src').src_in_domain_f1.mean().round(3).to_dict())

M = MT.groupby(['src', 'tgt', 'prior', 'method']).macro_f1.mean().unstack('method')
M['gap'] = M['oracle'] - M['source_only']
for m in ['bn_adapt', 'tent', 'shot']:
    M[f'rec_{m}'] = (M[m] - M['source_only']) / M['gap'].where(M['gap'].abs() > 1e-6)
print('\nGap and fraction of the gap recovered by each SFDA method:')
print(M.round(3).to_string())

U = TT[TT.method != 'oracle'].groupby(['tgt', 'method', 'seen_in_source']).recall.mean().unstack('seen_in_source').round(3)
print('\nMean attack-type recall on target (seen_in_source False = attack types the source never saw):'); print(U.to_string())
print('\nPer-type recall, natural prior:')
print(TT[(TT.prior == 'natural') & (TT.method != 'oracle')].groupby(['tgt', 'attack_type', 'seen_in_source', 'method']).recall.mean().unstack('method').round(3).to_string())

nat = M.xs('natural', level='prior')
gap_ok = bool((nat['gap'] >= CFG['gap_min']).any())
best_rec = nat[[f'rec_{m}' for m in ['bn_adapt', 'tent', 'shot']]].max(1)
weak_recovery = bool((best_rec < CFG['recovery_min']).any())
harm = [(i, m) for i, r in M.iterrows() for m in ['bn_adapt', 'tent', 'shot'] if r[m] < r['source_only'] - CFG['harm_margin']]
unseen = TT[(~TT.seen_in_source) & (TT.method.isin(['bn_adapt', 'tent', 'shot']))]
unseen_fail = bool(len(unseen) and unseen.groupby('method').recall.mean().max() < CFG['unseen_recall_min'])
verdict = dict(gap_ok=gap_ok, weak_recovery=weak_recovery, n_harm_cases=len(harm), harm_cases=[str(h) for h in harm][:20], unseen_fail=unseen_fail)
verdict['decision'] = ('GO' if gap_ok and (weak_recovery or len(harm) > 0 or unseen_fail)
                       else 'NO-GO: no meaningful cross-network gap' if not gap_ok
                       else 'NO-GO: existing SFDA already recovers the gap')
json.dump(verdict, open(f'{OUT}/verdict.json', 'w'), indent=2); print('\n', json.dumps(verdict, indent=2))